# Beacon Knapsack: direct vs. agentic orchestration

Both conditions use the same deterministic Beacon Knapsack recommender. The direct condition receives a selected persona. The agentic condition receives a normal-language request and must call three tools in order: `list_personas` → `get_persona` → `run_knapsack_for_persona`.

The LLM orchestrates; it does not choose food.

## Before running

Clone the companion Beacon backend and create its environment first. In a terminal, set `BEACON_BACKEND` to the backend directory containing `manage.py`. This notebook uses the backend's Python so it can call Beacon's actual Knapsack code and catalog.

The agentic cell asks for a Groq key only when enabled. The key remains in this notebook kernel for that run and is not written to a file.

In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'code' / 'benchmark.py').is_file():
    raise RuntimeError('Open this notebook from the repository root.')

CODE_DIR = PROJECT_ROOT / 'code'
RESULTS_DIR = PROJECT_ROOT / 'results'
BEACON_BACKEND = Path(os.environ.get('BEACON_BACKEND', '')).expanduser()
if not (BEACON_BACKEND / 'manage.py').is_file():
    raise RuntimeError('Set BEACON_BACKEND in a terminal, then restart the kernel.')

BEACON_PYTHON = BEACON_BACKEND / '.venv' / 'bin' / 'python'
if not BEACON_PYTHON.is_file():
    raise RuntimeError('Set up Beacon first, or change BEACON_PYTHON to its backend interpreter.')

print('Evaluation repository:', PROJECT_ROOT)
print('Beacon backend:', BEACON_BACKEND)
print('Python:', BEACON_PYTHON)

## 1. Verify the integration

These offline tests do not call an LLM. They verify that the adapter reaches Beacon's real Knapsack implementation, that its endpoint agrees for all six profiles, that plans are deterministic, and that Beacon's quality evaluators can score the returned plan.

In [ ]:
subprocess.run(
    [str(BEACON_PYTHON), '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=PROJECT_ROOT,
    check=True,
)

## 2. Run the direct deterministic reference

For each profile, Beacon Knapsack receives the stored restrictions, preferences, and requested meal roles. This produces the reference plan that the agentic condition must reproduce exactly.

In [ ]:
DIRECT_RESULT = RESULTS_DIR / 'direct_knapsack.json'
subprocess.run(
    [str(BEACON_PYTHON), str(CODE_DIR / 'benchmark.py'),
     '--condition', 'direct_knapsack', '--output', str(DIRECT_RESULT)],
    cwd=PROJECT_ROOT,
    check=True,
)

## 3. Optionally run the live agentic condition

Start with one profile. When enabled, the model receives a simple user request such as `Create a one-day meal plan for Jessica Smith.` The system instruction, tool schemas, and restricted tool session enforce the workflow. The benchmark records exact plan fidelity, tool sequence, retries, failures, and latency.

In [ ]:
RUN_AGENTIC_SMOKE_TEST = False  # Change to True only for a new live model call.

AGENTIC_RESULT = RESULTS_DIR / 'agentic_knapsack.json'
if RUN_AGENTIC_SMOKE_TEST:
    from getpass import getpass

    groq_key = getpass('Paste Groq API key for this run: ').strip()
    if not groq_key:
        raise ValueError('No API key entered.')
    run_environment = os.environ.copy()
    run_environment['GROQ_API_KEY'] = groq_key
    subprocess.run(
        [str(BEACON_PYTHON), str(CODE_DIR / 'benchmark.py'),
         '--condition', 'agentic_knapsack', '--limit', '1',
         '--output', str(AGENTIC_RESULT)],
        cwd=PROJECT_ROOT,
        env=run_environment,
        check=True,
    )
else:
    print('Live agentic smoke test skipped. Set RUN_AGENTIC_SMOKE_TEST = True to run it.')

## 4. Build a paired report

A comparison is valid only when direct and agentic results contain the same complete six-profile case set. The report generator labels a partial pair as a pilot rather than comparing its aggregate score. Nutrition-goal fit is post-hoc: it scores closeness between daily totals and stored goals, but is not an objective optimized by the current Knapsack selector.

In [ ]:
if DIRECT_RESULT.exists() and AGENTIC_RESULT.exists():
    TABLE_PATH = RESULTS_DIR / 'direct_vs_agentic.md'
    subprocess.run(
        [str(BEACON_PYTHON), str(CODE_DIR / 'report_results.py'),
         '--direct', str(DIRECT_RESULT), '--agentic', str(AGENTIC_RESULT),
         '--output', str(TABLE_PATH)],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print(TABLE_PATH.read_text(encoding='utf-8'))
else:
    print('Run both conditions before building a new paired report.')

## Interpretation

If direct and agentic plans match exactly, Knapsack meal selection has stayed constant. Any agentic failure then indicates an orchestration or provider failure, not a different Knapsack decision. See `results/final_knapsack_vs_agentic_quality.md` for the reviewed six-profile run included with this repository.